# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [4]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [5]:

EVENT_NAME = '202307_Flood_VT'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'planet'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [6]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = {
    "driver": "COG",
    "compress": "DEFLATE",
}

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 444 .tif files in the S3 bucket.


['drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144734_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144736_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144738_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144741_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144743_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144745_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144747_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144749_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144751_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_145051_colorInfrared.tif',
 'drcs_activations/2

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [8]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [10]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 16
  - Total size: 4.64 GB

📁 Cached files (first 10):
  - drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_144734_trueColor.tif (292.4 MB)
  - drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_144736_trueColor.tif (293.0 MB)
  - drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_144738_trueColor.tif (293.5 MB)
  - drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_144741_trueColor.tif (293.6 MB)
  - drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_144743_trueColor.tif (293.5 MB)
  - drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_144745_trueColor.tif (293.8 MB)
  - drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_144747_trueColor.tif (293.9 MB)
  - drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_144749_trueColor.tif (294.

(16, 4977485046)

In [11]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [12]:
keys

['drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144734_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144736_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144738_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144741_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144743_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144745_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144747_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144749_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_144751_colorInfrared.tif',
 'drcs_activations/202307_Flood_VT/planet/20230711/CIR/Planet_20230711_145051_colorInfrared.tif',
 'drcs_activations/2

# True color first (no New Hampshire)

In [13]:
# Define filename creator functions for different file types

def create_cog_filename_planet(f, EVENT_NAME):
    """Create COG filename for Planet files with event name first and date at end."""
    f2 = Path(f).stem
    parts = f2.split('_')
    
    # Find date part (YYYYMMDD format)
    date_index = None
    date_str = None
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_index = i
            date_str = part
            break
    
    if date_index is not None and date_str:
        # Format date
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Get parts before and after date
        prefix_parts = parts[:date_index]
        suffix_parts = parts[date_index + 1:]
        
        # Reconstruct: EVENT_NAME + prefix + suffix + date
        non_date_parts = prefix_parts + suffix_parts
        cog_filename = f'{EVENT_NAME}_{"_".join(non_date_parts)}_{formatted_date}day.tif'
    else:
        # No date found, just add event name
        cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    return cog_filename

pattern = re.compile(r'^(?!.*newHampshire).*/planet/.*trueColor.*\.tif$')

# Test functions
print("Testing WM filename:")
filter_ = [f for f in keys if pattern.match(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_planet(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202307_Flood_VT_Planet_144734_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144736_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144738_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144741_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144743_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144745_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144747_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144749_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144751_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_145051_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_145053_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_145056_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_145058_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_152046_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_152049_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_152051_trueColor_2023-07-11day.tif
  202307_Flood_VT_P

In [ ]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_planet, 
                                target_dir = "Planet/true", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202307_Flood_VT_Planet_144734_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144736_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144738_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144741_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144743_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144745_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144747_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144749_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_144751_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_145051_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_145053_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_145056_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_145058_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_152046_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_152049_trueColor_2023-07-11day.tif
  202307_Flood_VT_Planet_152051_trueColor_2023-07-11day.tif
  202307_Flood_VT_Pla

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...



   [MEMORY] High usage: 590.5 MB, forcing cleanup...
   [COGIFY] Creating COG from reprojected file...


   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Planet/true/202307_Flood_VT_Planet_144734_trueColor_2023-07-11day.tif
   [MEMORY] Final: 698.7 MB (Change: +407.5 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202307_Flood_VT_Planet_144734_trueColor_2023-07-11day.tif

[2/188] Processing: drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_144736_trueColor.tif
   Output filename: 202307_Flood_VT_Planet_144736_trueColor_2023-07-11day.tif
   [MEMORY] Initial: 698.7 MB
   [CACHE HIT] Using cached file: data_download/drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_144736_trueColor.tif
   [REPROJECT] Converting to EPSG:432

...

   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Planet/true/202307_Flood_VT_Planet_145058_trueColor_2023-07-11day.tif
   [MEMORY] Final: 783.1 MB (Change: +4.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202307_Flood_VT_Planet_145058_trueColor_2023-07-11day.tif

[14/188] Processing: drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_152046_trueColor.tif
   Output filename: 202307_Flood_VT_Planet_152046_trueColor_2023-07-11day.tif
   [MEMORY] Initial: 783.1 MB
   [CACHE HIT] Using cached file: data_download/drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_152046_tr

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Planet/true/202307_Flood_VT_Planet_152046_trueColor_2023-07-11day.tif
   [MEMORY] Final: 809.7 MB (Change: +26.6 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202307_Flood_VT_Planet_152046_trueColor_2023-07-11day.tif

[15/188] Processing: drcs_activations/202307_Flood_VT/planet/20230711/TrueColor/Planet_20230711_152049_trueColor.tif
   Output filename: 202307_Flood_VT_Planet_152049_trueColor_2023-07-11day.tif
   [MEMORY] Initial: 809.7 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache


In [ ]:
keys

# True color (with New Hampshire)

In [ ]:
# Define filename creator functions for different file types

def create_cog_filename_planet(f, EVENT_NAME):
    """Create COG filename for Planet files with event name first and date at end."""
    f2 = Path(f).stem
    parts = f2.split('_')
    
    # Find date part (YYYYMMDD format)
    date_index = None
    date_str = None
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_index = i
            date_str = part
            break
    
    if date_index is not None and date_str:
        # Format date
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Get parts before and after date
        prefix_parts = parts[:date_index]
        suffix_parts = parts[date_index + 1:]
        
        # Reconstruct: EVENT_NAME + prefix + suffix + date
        non_date_parts = prefix_parts + suffix_parts
        cog_filename = f'{EVENT_NAME}_{"_".join(non_date_parts)}_newHampshire_{formatted_date}day.tif'
    else:
        # No date found, just add event name
        cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    return cog_filename

pattern = re.compile(r'.*newHampshire.*trueColor.*\.tif$')

# Test functions
print("Testing WM filename:")
filter_ = [f for f in keys if pattern.match(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_planet(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




In [ ]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_planet, 
                                target_dir = "Planet/true", 
                                EVENT_NAME = EVENT_NAME)


# colorInfrared (no New Hampshire)

In [ ]:
# Define filename creator functions for different file types

def create_cog_filename_planet(f, EVENT_NAME):
    """Create COG filename for Planet files with event name first and date at end."""
    f2 = Path(f).stem
    parts = f2.split('_')
    
    # Find date part (YYYYMMDD format)
    date_index = None
    date_str = None
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_index = i
            date_str = part
            break
    
    if date_index is not None and date_str:
        # Format date
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Get parts before and after date
        prefix_parts = parts[:date_index]
        suffix_parts = parts[date_index + 1:]
        
        # Reconstruct: EVENT_NAME + prefix + suffix + date
        non_date_parts = prefix_parts + suffix_parts
        cog_filename = f'{EVENT_NAME}_{"_".join(non_date_parts)}_{formatted_date}day.tif'
    else:
        # No date found, just add event name
        cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    return cog_filename

pattern = re.compile(r'^(?!.*newHampshire).*/planet/.*colorInfrared.*\.tif$')

# Test functions
print("Testing WM filename:")
filter_ = [f for f in keys if pattern.match(f)]
print(filter_)
print()
print()

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_planet(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




In [ ]:
# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_planet, 
                                target_dir = "Planet/cir", 
                                EVENT_NAME = EVENT_NAME)

# colorInfrared (with New Hampshire)

In [ ]:
# Define filename creator functions for different file types

def create_cog_filename_planet(f, EVENT_NAME):
    """Create COG filename for Planet files with event name first and date at end."""
    f2 = Path(f).stem
    parts = f2.split('_')
    
    # Find date part (YYYYMMDD format)
    date_index = None
    date_str = None
    
    for i, part in enumerate(parts):
        if len(part) == 8 and part.isdigit() and part.startswith('20'):
            date_index = i
            date_str = part
            break
    
    if date_index is not None and date_str:
        # Format date
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Get parts before and after date
        prefix_parts = parts[:date_index]
        suffix_parts = parts[date_index + 1:]
        
        # Reconstruct: EVENT_NAME + prefix + suffix + date
        non_date_parts = prefix_parts + suffix_parts
        cog_filename = f'{EVENT_NAME}_{"_".join(non_date_parts)}_newHampshire_{formatted_date}day.tif'
    else:
        # No date found, just add event name
        cog_filename = f'{EVENT_NAME}_{f2}.tif'
    
    return cog_filename

pattern = re.compile(r'.*newHampshire.*colorInfrared.*\.tif$')

# Test functions
print("Testing WM filename:")
filter_ = [f for f in keys if pattern.match(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_planet(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




In [ ]:

# Process S1 WTR files
results2 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_planet, 
                                target_dir = "Planet/cir", 
                                EVENT_NAME = EVENT_NAME)

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")